In [ ]:
from config_dirs import CLEANING_INPUT, FINAL_OUTPUT

import sys
import pandas as pd
# To display all rows 
pd.set_option('display.max_rows', None)
# To display all columns
pd.set_option('display.max_columns', None)
# To prevent truncation of wide columns (full content of each column)
pd.set_option('display.max_colwidth', None)

Python version:
3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
Biopython version:
2.0.3


In [ ]:
# Load the Excel file into a DataFrame
gwas_to_clean_df = pd.read_csv(CLEANING_INPUT)

gwas_to_clean_df['PMID'].nunique()

# Extract unique PMIDs from the 'PMID' column and create a new dataframe
GWAS_unique_PMIDs = pd.DataFrame(gwas_to_clean_df['PMID'].unique(), columns=['PMID'])

# Display the first few rows of the new dataframe
GWAS_unique_PMIDs.head()

In [13]:
GWAS_unique_PMIDs.shape

(3778, 1)

In [ ]:
import pandas as pd
import time
import os
from Bio import Entrez

# Set your email and (optionally) your API key for Entrez
Entrez.email = "your_email@example.com"
# Entrez.api_key = "YOUR_NCBI_API_KEY"  # Uncomment and set if you have an API key

# Name of the file where results will be saved incrementally
output_filename = "pubmed_results.csv"

if os.path.exists(output_filename):
    print("Found an existing results file. Loading it to resume progress...")
    partial_df = pd.read_csv(output_filename)
    # Ensure the PMIDs are stored as strings for consistency.
    processed_pmids = set(partial_df['PMID'].astype(str))
    # Convert MeSH terms back from string representation if needed.
    results = partial_df.to_dict('records')
else:
    processed_pmids = set()
    results = []

# Total number of PMIDs to process
total_pmids = len(GWAS_unique_PMIDs)

# Iterate over each PMID from your unique PMIDs dataframe
for idx, row in GWAS_unique_PMIDs.iterrows():
    pmid = str(row['PMID'])
    
    # Skip PMIDs that have already been processed
    if pmid in processed_pmids:
        print(f"PMID {pmid} already processed. Skipping.")
        continue

    print(f"Processing PMID {pmid} ({idx + 1} of {total_pmids})")
    
    try:
        # Fetch the record from PubMed in XML format
        with Entrez.efetch(db="pubmed", id=pmid, retmode="xml") as handle:
            records = Entrez.read(handle)
        
        # Check that we got at least one article
        if records.get('PubmedArticle'):
            article = records['PubmedArticle'][0]
            medline = article.get('MedlineCitation', {})
            article_info = medline.get('Article', {})
            
            # --- Extract the Abstract ---
            abstract_text = None
            if "Abstract" in article_info and "AbstractText" in article_info["Abstract"]:
                abs_parts = article_info["Abstract"]["AbstractText"]
                # abs_parts may be a list or a single string. Combine if needed.
                if isinstance(abs_parts, list):
                    abstract_text = " ".join([str(part) for part in abs_parts])
                else:
                    abstract_text = str(abs_parts)
            
            # --- Extract the MeSH Terms ---
            mesh_terms_list = None
            if "MeshHeadingList" in medline:
                mesh_terms_list = []
                for mesh in medline["MeshHeadingList"]:
                    # Each mesh heading typically contains a DescriptorName element.
                    if "DescriptorName" in mesh:
                        mesh_terms_list.append(str(mesh["DescriptorName"]))
            # (If there are no MeshHeadingList entries, mesh_terms_list remains None.)
            
            # Append the result for this PMID
            results.append({
                "PMID": pmid,
                "Abstract_body": abstract_text,
                "MeshTerms": mesh_terms_list
            })
        else:
            # If no article was returned for this PMID
            results.append({
                "PMID": pmid,
                "Abstract_body": None,
                "MeshTerms": None
            })
        # Mark this PMID as processed
        processed_pmids.add(pmid)
    
    except Exception as e:
        print(f"Error processing PMID {pmid}: {e}")
        # In case of an error, wait a bit before continuing
        time.sleep(5)
        continue  # Skip to the next PMID
    
    # Sleep to avoid hitting the rate limit.
    # (NCBI recommends no more than 3 requests per second.)
    time.sleep(0.4)
    
    # Save intermediate results every 10 PMIDs (you can adjust this frequency)
    if (idx + 1) % 10 == 0:
        temp_df = pd.DataFrame(results)
        temp_df.to_csv(output_filename, index=False)
        print(f"Intermediate results saved after processing {idx + 1} records.")

# Save the final results once all PMIDs have been processed.
final_df = pd.DataFrame(results)
final_df.to_csv(FINAL_OUTPUT, index=False)
print(f"Finished processing all PMIDs. Final results saved to '{output_filename}'.")


Processing PMID 19060911 (1 of 3778)
Processing PMID 19074352 (2 of 3778)
Processing PMID 19103989 (3 of 3778)
Processing PMID 19108880 (4 of 3778)
Processing PMID 19136963 (5 of 3778)
Processing PMID 19151727 (6 of 3778)
Processing PMID 19179659 (7 of 3778)
Processing PMID 19198608 (8 of 3778)
Processing PMID 19198609 (9 of 3778)
Processing PMID 19198610 (10 of 3778)
Intermediate results saved after processing 10 records.
Processing PMID 19198611 (11 of 3778)
Processing PMID 19198612 (12 of 3778)
Processing PMID 19219041 (13 of 3778)
Processing PMID 19242469 (14 of 3778)
Processing PMID 19265035 (15 of 3778)
Processing PMID 19265041 (16 of 3778)
Processing PMID 19307479 (17 of 3778)
Processing PMID 19324915 (18 of 3778)
Processing PMID 19359598 (19 of 3778)
Processing PMID 19380626 (20 of 3778)
Intermediate results saved after processing 20 records.
Processing PMID 19390056 (21 of 3778)
Processing PMID 19398671 (22 of 3778)
Processing PMID 19411665 (23 of 3778)
Processing PMID 1941463

Processing PMID 17529978 (188 of 3778)
Processing PMID 17660820 (189 of 3778)
Processing PMID 17709650 (190 of 3778)
Intermediate results saved after processing 190 records.
Processing PMID 17846289 (191 of 3778)
Processing PMID 17967773 (192 of 3778)
Processing PMID 17994018 (193 of 3778)
Processing PMID 17996556 (194 of 3778)
Processing PMID 18084290 (195 of 3778)
Processing PMID 18086930 (196 of 3778)
Processing PMID 18176561 (197 of 3778)
Processing PMID 18193043 (198 of 3778)
Processing PMID 18235500 (199 of 3778)
Processing PMID 18268153 (200 of 3778)
Intermediate results saved after processing 200 records.
Processing PMID 18276926 (201 of 3778)
Processing PMID 18305085 (202 of 3778)
Processing PMID 18309101 (203 of 3778)
Processing PMID 18337603 (204 of 3778)
Processing PMID 18391953 (205 of 3778)
Processing PMID 18413308 (206 of 3778)
Processing PMID 18420941 (207 of 3778)
Processing PMID 18436795 (208 of 3778)
Processing PMID 18443590 (209 of 3778)
Processing PMID 18443592 (21

Processing PMID 16357305 (372 of 3778)
Processing PMID 16369530 (373 of 3778)
Processing PMID 16373601 (374 of 3778)
Processing PMID 16397141 (375 of 3778)
Processing PMID 16421173 (376 of 3778)
Processing PMID 16439692 (377 of 3778)
Processing PMID 16444274 (378 of 3778)
Processing PMID 16456101 (379 of 3778)
Processing PMID 16456111 (380 of 3778)
Intermediate results saved after processing 380 records.
Processing PMID 16461815 (381 of 3778)
Processing PMID 16484614 (382 of 3778)
Processing PMID 23804254 (383 of 3778)
Processing PMID 23825360 (384 of 3778)
Processing PMID 23852171 (385 of 3778)
Processing PMID 23868829 (386 of 3778)
Processing PMID 23913004 (387 of 3778)
Processing PMID 23955597 (388 of 3778)
Processing PMID 23969695 (389 of 3778)
Processing PMID 23973703 (390 of 3778)
Intermediate results saved after processing 390 records.
Processing PMID 24014829 (391 of 3778)
Processing PMID 24185693 (392 of 3778)
Processing PMID 24201301 (393 of 3778)
Processing PMID 24213632 (39

Processing PMID 11349008 (556 of 3778)
Processing PMID 11350093 (557 of 3778)
Processing PMID 11369687 (558 of 3778)
Processing PMID 11425780 (559 of 3778)
Processing PMID 11433350 (560 of 3778)
Intermediate results saved after processing 560 records.
Processing PMID 11447084 (561 of 3778)
Processing PMID 11474113 (562 of 3778)
Processing PMID 11484688 (563 of 3778)
Processing PMID 11484689 (564 of 3778)
Processing PMID 11498583 (565 of 3778)
Processing PMID 11499717 (566 of 3778)
Processing PMID 11502697 (567 of 3778)
Processing PMID 11502704 (568 of 3778)
Processing PMID 11509445 (569 of 3778)
Processing PMID 11514372 (570 of 3778)
Intermediate results saved after processing 570 records.
Processing PMID 11532900 (571 of 3778)
Processing PMID 11557743 (572 of 3778)
Processing PMID 11560853 (573 of 3778)
Processing PMID 11568057 (574 of 3778)
Processing PMID 11568073 (575 of 3778)
Processing PMID 11568074 (576 of 3778)
Processing PMID 11568079 (577 of 3778)
Processing PMID 21307941 (57

Processing PMID 33793338 (739 of 3778)
Processing PMID 33885752 (740 of 3778)
Intermediate results saved after processing 740 records.
Processing PMID 33938943 (741 of 3778)
Processing PMID 33950175 (742 of 3778)
Processing PMID 33998272 (743 of 3778)
Processing PMID 34111939 (744 of 3778)
Processing PMID 34129019 (745 of 3778)
Processing PMID 34137813 (746 of 3778)
Processing PMID 34182767 (747 of 3778)
Processing PMID 34325833 (748 of 3778)
Processing PMID 34385711 (749 of 3778)
Processing PMID 34424322 (750 of 3778)
Intermediate results saved after processing 750 records.
Processing PMID 34427584 (751 of 3778)
Processing PMID 34433968 (752 of 3778)
Processing PMID 34464975 (753 of 3778)
Processing PMID 34471932 (754 of 3778)
Processing PMID 34496175 (755 of 3778)
Processing PMID 34662886 (756 of 3778)
Processing PMID 34788059 (757 of 3778)
Processing PMID 34788399 (758 of 3778)
Processing PMID 34837083 (759 of 3778)
Processing PMID 34852172 (760 of 3778)
Intermediate results saved a

...

Processing PMID 28662310 (2902 of 3778)
Processing PMID 28666732 (2903 of 3778)
Processing PMID 28671346 (2904 of 3778)
Processing PMID 28677864 (2905 of 3778)
Processing PMID 28709802 (2906 of 3778)
Processing PMID 28734077 (2907 of 3778)
Processing PMID 28737171 (2908 of 3778)
Processing PMID 28740544 (2909 of 3778)
Processing PMID 28750931 (2910 of 3778)
Intermediate results saved after processing 2910 records.
Processing PMID 28754818 (2911 of 3778)
Processing PMID 28761118 (2912 of 3778)
Processing PMID 28771277 (2913 of 3778)
Processing PMID 28775000 (2914 of 3778)
Processing PMID 28911789 (2915 of 3778)
Processing PMID 28923913 (2916 of 3778)
Processing PMID 28924240 (2917 of 3778)
Processing PMID 28931820 (2918 of 3778)
Processing PMID 28937974 (2919 of 3778)
Processing PMID 28951125 (2920 of 3778)
Intermediate results saved after processing 2920 records.
Processing PMID 28951494 (2921 of 3778)
Processing PMID 28958774 (2922 of 3778)
Processing PMID 28969386 (2923 of 3778)
Proc

Processing PMID 29563136 (3082 of 3778)
Processing PMID 29563506 (3083 of 3778)
Processing PMID 29603773 (3084 of 3778)
Processing PMID 29606302 (3085 of 3778)
Processing PMID 29625025 (3086 of 3778)
Processing PMID 29631062 (3087 of 3778)
Processing PMID 29631342 (3088 of 3778)
Processing PMID 29653195 (3089 of 3778)
Processing PMID 29654271 (3090 of 3778)
Intermediate results saved after processing 3090 records.
Processing PMID 29661775 (3091 of 3778)
Processing PMID 29664015 (3092 of 3778)
Processing PMID 29688154 (3093 of 3778)
Processing PMID 29695719 (3094 of 3778)
Processing PMID 29703722 (3095 of 3778)
Processing PMID 29703982 (3096 of 3778)
Processing PMID 29709596 (3097 of 3778)
Processing PMID 29847840 (3098 of 3778)
Processing PMID 29850784 (3099 of 3778)
Processing PMID 29858212 (3100 of 3778)
Intermediate results saved after processing 3100 records.
Processing PMID 29863498 (3101 of 3778)
Processing PMID 29864785 (3102 of 3778)
Processing PMID 29864786 (3103 of 3778)
Proc

Processing PMID 32989217 (3262 of 3778)
Processing PMID 33000479 (3263 of 3778)
Processing PMID 33001231 (3264 of 3778)
Processing PMID 33031748 (3265 of 3778)
Processing PMID 33037005 (3266 of 3778)
Processing PMID 33052245 (3267 of 3778)
Processing PMID 33052496 (3268 of 3778)
Processing PMID 33054575 (3269 of 3778)
Processing PMID 33057009 (3270 of 3778)
Intermediate results saved after processing 3270 records.
Processing PMID 33058850 (3271 of 3778)
Processing PMID 33090602 (3272 of 3778)
Processing PMID 33104469 (3273 of 3778)
Processing PMID 33107092 (3274 of 3778)
Processing PMID 33110216 (3275 of 3778)
Processing PMID 33111990 (3276 of 3778)
Processing PMID 33218968 (3277 of 3778)
Processing PMID 33246059 (3278 of 3778)
Processing PMID 33247606 (3279 of 3778)
Processing PMID 33248215 (3280 of 3778)
Intermediate results saved after processing 3280 records.
Processing PMID 33273682 (3281 of 3778)
Processing PMID 33280500 (3282 of 3778)
Processing PMID 33285523 (3283 of 3778)
Proc

Processing PMID 33742435 (3442 of 3778)
Processing PMID 33757580 (3443 of 3778)
Processing PMID 33767199 (3444 of 3778)
Processing PMID 33854038 (3445 of 3778)
Processing PMID 33859766 (3446 of 3778)
Processing PMID 33875642 (3447 of 3778)
Processing PMID 33891857 (3448 of 3778)
Processing PMID 33892507 (3449 of 3778)
Processing PMID 33895794 (3450 of 3778)
Intermediate results saved after processing 3450 records.
Processing PMID 33897872 (3451 of 3778)
Processing PMID 33904890 (3452 of 3778)
Processing PMID 33905372 (3453 of 3778)
Processing PMID 33906245 (3454 of 3778)
Processing PMID 33909933 (3455 of 3778)
Processing PMID 33914953 (3456 of 3778)
Processing PMID 33938457 (3457 of 3778)
Processing PMID 34043589 (3458 of 3778)
Processing PMID 34048683 (3459 of 3778)
Processing PMID 34077955 (3460 of 3778)
Intermediate results saved after processing 3460 records.
Processing PMID 34099646 (3461 of 3778)
Processing PMID 34102396 (3462 of 3778)
Processing PMID 34115836 (3463 of 3778)
Proc

Processing PMID 37816352 (3622 of 3778)
Processing PMID 37828025 (3623 of 3778)
Processing PMID 37830409 (3624 of 3778)
Processing PMID 37832608 (3625 of 3778)
Processing PMID 37852978 (3626 of 3778)
Processing PMID 37907055 (3627 of 3778)
Processing PMID 37919320 (3628 of 3778)
Processing PMID 37923132 (3629 of 3778)
Processing PMID 37925478 (3630 of 3778)
Intermediate results saved after processing 3630 records.
Processing PMID 37931331 (3631 of 3778)
Processing PMID 37942644 (3632 of 3778)
Processing PMID 37955334 (3633 of 3778)
Processing PMID 37978175 (3634 of 3778)
Processing PMID 37989727 (3635 of 3778)
Processing PMID 38001255 (3636 of 3778)
Processing PMID 38011863 (3637 of 3778)
Processing PMID 38030619 (3638 of 3778)
Processing PMID 38085578 (3639 of 3778)
Processing PMID 38086820 (3640 of 3778)
Intermediate results saved after processing 3640 records.
Processing PMID 35088838 (3641 of 3778)
Processing PMID 35943854 (3642 of 3778)
Processing PMID 35984902 (3643 of 3778)
Proc

In [ ]:
import pandas as pd

# Load the full output CSV
final_df = pd.read_csv(output_filename)

# -------------------------------
# 1. Filter for cardiovascular-related entries
# - Keep rows where 'Causality' contains "Causal"
# - Keep rows where 'Organ/System' contains 'Cardiovascular' (case-insensitive) or 'vascular'
# -------------------------------
cardio_df = final_df[
    final_df['Causality'].str.contains('Causal', case=False, na=False) &
    final_df['Organ/System'].str.contains('Cardiovascular|vascular', case=False, na=False)
]
# Save cardiovascular filtered data
cardio_df.to_csv('cardiovascular.csv', index=False)
print("Cardiovascular filtered data saved to 'cardiovascular.csv'")

# -------------------------------
# 2. Filter for brain-related entries
# - Keep rows where 'Causality' contains "Causal"
# - Keep rows where 'Organ/System' contains 'Brain', 'nervous system' or 'cerebrovascular' (case-insensitive)
# -------------------------------
brain_df = final_df[
    final_df['Causality'].str.contains('Causal', case=False, na=False) &
    final_df['Organ/System'].str.contains('Brain|nervous system|cerebrovascular', case=False, na=False)
]
# Save brain filtered data
brain_df.to_csv('brain.csv', index=False)
print("Brain filtered data saved to 'brain.csv'")

# -------------------------------
# 3. Find shared genes between cardiovascular and brain
# - Using the 'Perplexity' column to identify common genes
# - Keep all other columns from each dataset
# -------------------------------
shared_genes_df = pd.merge(
    cardio_df, 
    brain_df, 
    on='Perplexity',  # Intersection based on Perplexity (gene name)
    suffixes=('_cardio', '_brain')
)

# Save the intersection to Excel
shared_genes_df.to_excel('Shared_genes_final.xlsx', index=False)
print("Shared genes saved to 'Shared_genes_final.xlsx'")
